In [ ]:
import os

os.environ['HTTP_PROXY'] = "http://127.0.0.1:1081"
os.environ['HTTPS_PROXY'] = "http://127.0.0.1:1081"

# with vLLM 0.8, need this to access the true model with model_executor
os.environ['VLLM_USE_V1'] = '0'

In [2]:
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.8)

# Create an LLM.
llm = LLM(model=model_name, enable_sleep_mode=True)
# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
results = llm.generate(prompts, sampling_params)
# Print the outputs.
for item in results:
    prompt = item.prompt
    generated_text = item.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

/home/michael/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-05 18:59:04 [__init__.py:239] Automatically detected platform cuda.


2025-04-05 18:59:04,228	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-05 18:59:12 [config.py:585] This model supports multiple tasks: {'classify', 'embed', 'generate', 'reward', 'score'}. Defaulting to 'generate'.
INFO 04-05 18:59:12 [llm_engine.py:241] Initializing a V0 LLM engine (v0.8.2) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.97it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.97it/s]


...

INFO 04-05 18:59:18 [worker.py:267] Memory profiling takes 1.00 seconds
INFO 04-05 18:59:18 [worker.py:267] the current vLLM instance can use total_gpu_memory (23.57GiB) x gpu_memory_utilization (0.90) = 21.21GiB
INFO 04-05 18:59:18 [worker.py:267] model weights take 0.93GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.44GiB; the rest of the memory reserved for KV Cache is 18.79GiB.
INFO 04-05 18:59:18 [executor_base.py:111] # cuda blocks: 102604, # CPU blocks: 21845
INFO 04-05 18:59:18 [executor_base.py:116] Maximum concurrency for 32768 tokens per request: 50.10x
INFO 04-05 18:59:23 [model_runner.py:1442] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_nu

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:24<00:00,  1.45it/s]

INFO 04-05 18:59:47 [model_runner.py:1570] Graph capturing finished in 24 secs, took 0.15 GiB
INFO 04-05 18:59:47 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 30.54 seconds



Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 67.98it/s, est. speed input: 374.90 toks/s, output: 1090.54 toks/s]

Prompt: 'Hello, my name is', Generated text: " Emily. I'm 12 years old, and I like to write my"
Prompt: 'The president of the United States is', Generated text: ' facing a big decision on how to handle a sudden global pandemic. He has been'
Prompt: 'The capital of France is', Generated text: ' Paris and the capital of Switzerland is Zurich.\nA. 错误\nB'
Prompt: 'The future of AI is', Generated text: ' still uncertain, with risks abounding. A certain experimental project uses a supercomputer'


In [3]:
# move to CPU
llm.sleep(level=1)

INFO 04-05 18:59:50 [worker.py:133] Sleep mode freed 19.77 GiB memory, 0.53 GiB memory is still in use.
INFO 04-05 18:59:50 [executor_base.py:208] It took 0.925358 seconds to fall asleep.


In [4]:
# move to GPU
llm.wake_up()

INFO 04-05 18:59:51 [executor_base.py:219] It took 0.410512 seconds to wake up.


In [5]:
# sampling_params = SamplingParams(temperature=0.8, max_tokens=512, skip_special_tokens=False, include_stop_str_in_output=True)
sampling_params = SamplingParams(temperature=0.8, max_tokens=512)

chat_prompts = [
    '<|im_start|>user\nIn a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?<|im_end|>\n<|im_start|>assistant\n'
]

results = llm.generate(chat_prompts, sampling_params)
# Print the outputs.
for item in results:
    prompt = item.prompt
    prompt_ids = item.prompt_token_ids
    generated_ids = item.outputs[0].token_ids
    generated_text = item.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")
    print('\n\n')
    print(f"Generated ids [{len(generated_ids)}]: {generated_ids}")

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s, est. speed input: 71.17 toks/s, output: 399.72 toks/s]

Prompt: '<|im_start|>user\nIn a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?<|im_end|>\n<|im_start|>assistant\n', Generated text: "To determine the percentage of the entire dance class enrolled in hip-hop dance, we can follow these steps:\n\n1. Calculate the number of students enrolled in contemporary and jazz dance.\n2. Subtract the number of students enrolled in contemporary and jazz dance from the total number of students to find out how many are enrolled in hip-hop dance.\n3. Calculate the percentage of hip-hop dance students.\n\nLet's start with the first step:\n\n1. The total number of students is 20. We need to find out how many of them are enrolled in contemporary dance. Since 20% of the students are in contemporary dance, we calculate:\n   \\[\n   0.20 \\times 20 = 4 \\text{ students}\n   \\]\n   So, 4 student

## Update engine weights

In [9]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    PreTrainedModel,
    PreTrainedTokenizer,
)

train_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
import torch

new_state_dict = train_model.state_dict()

In [13]:
native_model = llm.llm_engine.model_executor.driver_worker.model_runner.model

native_model.load_weights(new_state_dict.items())

{'model.embed_tokens.weight',
 'model.layers.0.input_layernorm.weight',
 'model.layers.0.mlp.down_proj.weight',
 'model.layers.0.mlp.gate_up_proj.weight',
 'model.layers.0.post_attention_layernorm.weight',
 'model.layers.0.self_attn.o_proj.weight',
 'model.layers.0.self_attn.qkv_proj.bias',
 'model.layers.0.self_attn.qkv_proj.weight',
 'model.layers.1.input_layernorm.weight',
 'model.layers.1.mlp.down_proj.weight',
 'model.layers.1.mlp.gate_up_proj.weight',
 'model.layers.1.post_attention_layernorm.weight',
 'model.layers.1.self_attn.o_proj.weight',
 'model.layers.1.self_attn.qkv_proj.bias',
 'model.layers.1.self_attn.qkv_proj.weight',
 'model.layers.10.input_layernorm.weight',
 'model.layers.10.mlp.down_proj.weight',
 'model.layers.10.mlp.gate_up_proj.weight',
 'model.layers.10.post_attention_layernorm.weight',
 'model.layers.10.self_attn.o_proj.weight',
 'model.layers.10.self_attn.qkv_proj.bias',
 'model.layers.10.self_attn.qkv_proj.weight',
 'model.layers.11.input_layernorm.weight',

Using the new method seems ends with error where weights keys are different

In [10]:


# Assume llm is your vLLM LLM instance and new_state_dict is a PyTorch state dict of updated weights.
def update_model_weights(model: torch.nn.Module):
    # Option 1: Use standard PyTorch method to load the full state dict
    model.load_state_dict(new_state_dict)
    # Option 2: (Advanced) Use vLLM's internal load_weights for partial updates:
    # model.load_weights(weights=list(new_state_dict.items()))
    return "OK"  # (optional) return a result or None

# Apply the update function to the vLLM model. This runs on the internal model object.
result = llm.apply_model(update_model_weights)
print(result)  # In single-GPU mode, this will print something like ['OK']


RuntimeError: Error(s) in loading state_dict for Qwen2ForCausalLM:
	Missing key(s) in state_dict: "model.layers.0.self_attn.qkv_proj.weight", "model.layers.0.self_attn.qkv_proj.bias", "model.layers.0.mlp.gate_up_proj.weight", "model.layers.1.self_attn.qkv_proj.weight", "model.layers.1.self_attn.qkv_proj.bias", "model.layers.1.mlp.gate_up_proj.weight", "model.layers.2.self_attn.qkv_proj.weight", "model.layers.2.self_attn.qkv_proj.bias", "model.layers.2.mlp.gate_up_proj.weight", "model.layers.3.self_attn.qkv_proj.weight", "model.layers.3.self_attn.qkv_proj.bias", "model.layers.3.mlp.gate_up_proj.weight", "model.layers.4.self_attn.qkv_proj.weight", "model.layers.4.self_attn.qkv_proj.bias", "model.layers.4.mlp.gate_up_proj.weight", "model.layers.5.self_attn.qkv_proj.weight", "model.layers.5.self_attn.qkv_proj.bias", "model.layers.5.mlp.gate_up_proj.weight", "model.layers.6.self_attn.qkv_proj.weight", "model.layers.6.self_attn.qkv_proj.bias", "model.layers.6.mlp.gate_up_proj.weight", "model